In [1]:
import boto3
from os import getenv
import json
from pprint import pprint
import awswrangler as wr
import pandas as pd
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
%matplotlib inline


In [2]:
session = boto3.Session(profile_name='SA', region_name='eu-central-1')

In [3]:
household = '26802fbe-7b56-467f-8488-fd5f9e1482de'  # Julia

In [28]:
SQL = f"""SELECT h.id, p4.*
FROM usage.p4_hour_2025 p4
JOIN usage.vw_households h ON p4.meter_ean = h.gas_ean
WHERE h.id = '{household}'
AND p4.date >= '2025-05-01'
AND p4.date < '2026-01-01'
AND p4.type = 'gas'
ORDER BY p4.date"""

In [29]:
df_wide = wr.athena.read_sql_query(
    sql=SQL,
    database="usage",
    s3_output="s3://slimwonen-athena-queries/",
    workgroup="primary",
    boto3_session=session,
)

In [30]:
def wide_to_long(df: pd.DataFrame) -> pd.DataFrame:
    hour_cols = [c for c in df.columns if c.startswith('measurement_h_')]
    
    long = df.melt(
        id_vars=['date'],
        value_vars=hour_cols,
        var_name='hour',
        value_name='reading'
    )
    
    long['hour'] = long['hour'].str.extract(r'(\d+)').astype(int)
    long['datetime'] = pd.to_datetime(long['date']) + pd.to_timedelta(long['hour'], unit='h')
    long['date'] = long['datetime'].dt.date
    long['time'] = long['datetime'].dt.time
    
    return (
        long[['date', 'time', 'datetime', 'reading']]
        .sort_values('datetime')
        .reset_index(drop=True)
    )
df_long = wide_to_long(df_wide)
df_long.drop_duplicates(inplace=True)
df_long['gap_hours'] = df_long['datetime'].diff().dt.total_seconds() / 3600
df_long['usage'] = df_long['reading'].diff().round(3)
df_long.loc[df_long['gap_hours'] != 1, 'usage'] = None  # nullify usage across gaps

In [31]:
df_long[pd.isna(df_long['usage'])]

,date,time,datetime,reading,gap_hours,usage
0,2025-05-01,00:00:00,2025-05-01,980.624,NaN,NaN
650,2025-05-28,00:00:00,2025-05-28,997.893,24.0,NaN
950,2025-06-10,00:00:00,2025-06-10,1002.933,24.0,NaN
1025,2025-06-14,00:00:00,2025-06-14,1003.811,24.0,NaN
3925,2025-10-09,00:00:00,2025-10-09,1041.271,24.0,NaN
4025,2025-10-14,00:00:00,2025-10-14,1043.243,24.0,NaN
4075,2025-10-17,00:00:00,2025-10-17,1044.480,24.0,NaN


In [33]:
df_long.to_csv('julia_gas_usage.csv', index=False)